# EA Sports Player Performance Index


## Import libraries

In [1]:
import math
import sys
import warnings
from pathlib import Path

import pandas as pd
import socceraction.spadl as spadl
from socceraction.data.statsbomb import StatsBombLoader

In [2]:
# Add the project root to the Python path
sys.path.append(str(Path.cwd().parents[1]))
from config import project_paths

In [3]:
# Ignore specified warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings(
    "ignore",
    message="Inferred xy_fidelity_version=2. If this is incorrect, please specify the correct version using the xy_fidelity_version argument",
    category=UserWarning,
)

## Load providers data

In [4]:
# StatsBomb match and player IDs for testing (UEFA Euro 2024 Final - Lamine Yamal)
COMPETITION_ID = 55
SEASON_ID = 282
MATCH_ID = 3943043
PLAYER_ID = 316046

In [5]:
SBL = StatsBombLoader(getter="local", root=str(project_paths.STATSBOMB_DIR))

## Data exploration

In [6]:
# Load StatsBomb data for the specified match
df_events = SBL.events(game_id=MATCH_ID)

In [7]:
# Display first 5 rows of the StatsBomb DataFrame
df_events.head()

,game_id,event_id,period_id,team_id,player_id,type_id,type_name,index,timestamp,minute,...,team_name,duration,extra,related_events,player_name,position_id,position_name,location,under_pressure,counterpress
0,3943043,50aa204f-5d65-4145-8597-5d5628fb7898,1,772,NaN,35,Starting XI,1,0 days 00:00:00,0,...,Spain,0.000000,"{'tactics': {'formation': 4231, 'lineup': [{'p...",[],NaN,NaN,NaN,NaN,False,False
1,3943043,a279cbee-9ab3-4cfb-9c51-27cacc1bf2a2,1,768,NaN,35,Starting XI,2,0 days 00:00:00,0,...,England,0.000000,"{'tactics': {'formation': 4231, 'lineup': [{'p...",[],NaN,NaN,NaN,NaN,False,False
2,3943043,d2126e70-9f04-4bb7-ba2b-9377836d1757,1,768,NaN,18,Half Start,3,0 days 00:00:00,0,...,England,0.000000,{},[54d78bfa-4146-42bd-acdc-97bcd393dd81],NaN,NaN,NaN,NaN,False,False
3,3943043,54d78bfa-4146-42bd-acdc-97bcd393dd81,1,772,NaN,18,Half Start,4,0 days 00:00:00,0,...,Spain,0.000000,{},[d2126e70-9f04-4bb7-ba2b-9377836d1757],NaN,NaN,NaN,NaN,False,False
4,3943043,152820f0-6ca9-4df3-943b-a67d568ff472,1,768,99174.0,30,Pass,5,0 days 00:00:00.340000,0,...,England,2.529454,"{'pass': {'recipient': {'id': 3468, 'name': 'J...",[d64668c7-747c-4a7d-912c-e1c3ff357a67],Kobbie Mainoo,9.0,Right Defensive Midfield,"[60.0, 40.0]",False,False


## Convert to SPADL

In [8]:
# Load games for the specified competition and season
df_games = SBL.games(competition_id=COMPETITION_ID, season_id=SEASON_ID)

In [9]:
# Obtain home_team_id for the specified match
home_team_id = df_games[df_games["game_id"] == MATCH_ID]["home_team_id"].values[0]

In [10]:
# Convert events to actions in SPADL format
df_actions = spadl.statsbomb.convert_to_actions(df_events, home_team_id=home_team_id)

In [11]:
# Display SPADL actions DataFrame
df_actions.head()

,game_id,original_event_id,period_id,time_seconds,team_id,player_id,start_x,start_y,end_x,end_y,type_id,result_id,bodypart_id,action_id
0,3943043,152820f0-6ca9-4df3-943b-a67d568ff472,1,0.340,768,99174.0,52.54375,33.9575,82.81875,32.9375,0,1,5,0
1,3943043,9c107df3-a3c8-4ad5-bc35-00214087a105,1,2.870,768,3468.0,82.81875,32.9375,79.93125,26.8175,21,1,0,1
2,3943043,237201b8-aef8-4823-b282-e82875795c07,1,4.742,768,3468.0,79.93125,26.8175,0.04375,57.5025,0,0,4,2
3,3943043,238f44cb-0f18-4217-85b5-8cc6345278fe,1,34.440,772,11748.0,5.99375,34.3825,7.91875,19.4225,22,1,4,3
4,3943043,987a3f1e-4dc4-4833-8896-5513e35c3fc5,1,35.658,772,22128.0,7.91875,19.4225,7.74375,19.4225,21,1,0,4


In [12]:
df_actions = (
    spadl.add_names(df_actions)  # Add the type name, result name and bodypart name
    .merge(SBL.teams(game_id=MATCH_ID))  # Add team names
    .merge(SBL.players(game_id=MATCH_ID))  # Add player names
)

In [13]:
df_actions.head()

,game_id,original_event_id,period_id,time_seconds,team_id,player_id,start_x,start_y,end_x,end_y,...,result_name,bodypart_name,team_name,player_name,nickname,jersey_number,is_starter,starting_position_id,starting_position_name,minutes_played
0,3943043,152820f0-6ca9-4df3-943b-a67d568ff472,1,0.340,768,99174.0,52.54375,33.9575,82.81875,32.9375,...,success,foot_right,England,Kobbie Mainoo,None,26,True,9,Right Defensive Midfield,71
1,3943043,9c107df3-a3c8-4ad5-bc35-00214087a105,1,2.870,768,3468.0,82.81875,32.9375,79.93125,26.8175,...,success,foot,England,Jordan Pickford,None,1,True,1,Goalkeeper,96
2,3943043,237201b8-aef8-4823-b282-e82875795c07,1,4.742,768,3468.0,79.93125,26.8175,0.04375,57.5025,...,fail,foot_left,England,Jordan Pickford,None,1,True,1,Goalkeeper,96
3,3943043,238f44cb-0f18-4217-85b5-8cc6345278fe,1,34.440,772,11748.0,5.99375,34.3825,7.91875,19.4225,...,success,foot_left,Spain,Unai Simón Mendibil,Unai Simón,23,True,1,Goalkeeper,96
4,3943043,987a3f1e-4dc4-4833-8896-5513e35c3fc5,1,35.658,772,22128.0,7.91875,19.4225,7.74375,19.4225,...,success,foot,Spain,Robin Aime Robert Le Normand,Robin Le Normand,3,True,3,Right Center Back,84


In [14]:
df_actions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1883 entries, 0 to 1882
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   game_id                 1883 non-null   int64  
 1   original_event_id       1857 non-null   object 
 2   period_id               1883 non-null   int64  
 3   time_seconds            1883 non-null   float64
 4   team_id                 1883 non-null   int64  
 5   player_id               1883 non-null   float64
 6   start_x                 1883 non-null   float64
 7   start_y                 1883 non-null   float64
 8   end_x                   1883 non-null   float64
 9   end_y                   1883 non-null   float64
 10  type_id                 1883 non-null   int64  
 11  result_id               1883 non-null   int64  
 12  bodypart_id             1883 non-null   int64  
 13  action_id               1883 non-null   int64  
 14  type_name               1883 non-null   

## Filter columns from the dataset

In [15]:
# Filter relevant columns for analysis
filtered_df = df_actions[
    [
        # Player
        "player_id",
        "player_name",
        "nickname",
        "minutes_played",
        "is_starter",
        "starting_position_name",
        # Team
        "team_id",
        "team_name",
        # Event
        "original_event_id",
        "type_name",
        "result_name",
        "bodypart_name",
        "period_id",
        "time_seconds",
        "start_x",
        "start_y",
        "end_x",
        "end_y",
    ]
]

# Display the filtered DataFrame
filtered_df.head()

,player_id,player_name,nickname,minutes_played,is_starter,starting_position_name,team_id,team_name,original_event_id,type_name,result_name,bodypart_name,period_id,time_seconds,start_x,start_y,end_x,end_y
0,99174.0,Kobbie Mainoo,None,71,True,Right Defensive Midfield,768,England,152820f0-6ca9-4df3-943b-a67d568ff472,pass,success,foot_right,1,0.340,52.54375,33.9575,82.81875,32.9375
1,3468.0,Jordan Pickford,None,96,True,Goalkeeper,768,England,9c107df3-a3c8-4ad5-bc35-00214087a105,dribble,success,foot,1,2.870,82.81875,32.9375,79.93125,26.8175
2,3468.0,Jordan Pickford,None,96,True,Goalkeeper,768,England,237201b8-aef8-4823-b282-e82875795c07,pass,fail,foot_left,1,4.742,79.93125,26.8175,0.04375,57.5025
3,11748.0,Unai Simón Mendibil,Unai Simón,96,True,Goalkeeper,772,Spain,238f44cb-0f18-4217-85b5-8cc6345278fe,goalkick,success,foot_left,1,34.440,5.99375,34.3825,7.91875,19.4225
4,22128.0,Robin Aime Robert Le Normand,Robin Le Normand,84,True,Right Center Back,772,Spain,987a3f1e-4dc4-4833-8896-5513e35c3fc5,dribble,success,foot,1,35.658,7.91875,19.4225,7.74375,19.4225


In [16]:
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1883 entries, 0 to 1882
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   player_id               1883 non-null   float64
 1   player_name             1883 non-null   object 
 2   nickname                1021 non-null   object 
 3   minutes_played          1883 non-null   int64  
 4   is_starter              1883 non-null   bool   
 5   starting_position_name  1883 non-null   object 
 6   team_id                 1883 non-null   int64  
 7   team_name               1883 non-null   object 
 8   original_event_id       1857 non-null   object 
 9   type_name               1883 non-null   object 
 10  result_name             1883 non-null   object 
 11  bodypart_name           1883 non-null   object 
 12  period_id               1883 non-null   int64  
 13  time_seconds            1883 non-null   float64
 14  start_x                 1883 non-null   

## Group events by player

In [17]:
# Count events by player
event_count_by_player_df = filtered_df.groupby(["player_name", "type_name"]).size().unstack(fill_value=0)

# Remove columns axis name and convert "player" from index to a column
event_count_by_player_df = event_count_by_player_df.rename_axis(None, axis="columns").reset_index()

# Show head of DataFrame
event_count_by_player_df.head(10)

,player_name,bad_touch,clearance,corner_crossed,corner_short,cross,dribble,foul,freekick_crossed,freekick_short,goalkick,interception,keeper_claim,keeper_punch,keeper_save,pass,shot,tackle,take_on,throw_in
0,Aymeric Laporte,0,7,0,0,0,73,0,0,1,2,0,0,0,0,80,1,0,0,0
1,Bukayo Saka,0,1,0,0,2,30,2,0,0,0,0,0,0,0,22,0,1,1,0
2,Cole Palmer,0,1,1,0,0,8,0,0,1,0,0,0,0,0,5,1,1,1,0
3,Daniel Carvajal Ramos,0,2,0,0,2,53,1,0,0,0,2,0,0,0,57,0,4,2,14
4,Daniel Olmo Carvajal,2,0,0,0,2,32,1,0,0,0,1,0,0,0,30,2,1,1,0
5,Declan Rice,0,0,0,0,0,35,1,1,1,0,3,0,0,0,40,3,2,2,0
6,Fabián Ruiz Peña,0,0,0,0,0,60,0,0,1,0,0,0,0,0,62,2,4,1,1
7,Harry Kane,0,0,0,0,0,7,1,0,0,0,0,0,0,0,11,1,0,0,0
8,John Stones,0,8,0,0,0,27,1,1,2,0,0,0,0,0,32,0,1,0,0
9,Jordan Pickford,0,0,0,0,0,25,0,2,0,11,0,3,2,4,29,0,0,0,0


In [18]:
# Display events info
event_count_by_player_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   player_name       28 non-null     object
 1   bad_touch         28 non-null     int64 
 2   clearance         28 non-null     int64 
 3   corner_crossed    28 non-null     int64 
 4   corner_short      28 non-null     int64 
 5   cross             28 non-null     int64 
 6   dribble           28 non-null     int64 
 7   foul              28 non-null     int64 
 8   freekick_crossed  28 non-null     int64 
 9   freekick_short    28 non-null     int64 
 10  goalkick          28 non-null     int64 
 11  interception      28 non-null     int64 
 12  keeper_claim      28 non-null     int64 
 13  keeper_punch      28 non-null     int64 
 14  keeper_save       28 non-null     int64 
 15  pass              28 non-null     int64 
 16  shot              28 non-null     int64 
 17  tackle            

## Constants

In [19]:
# COEFFICIENTS FOR THE INDEX

## Subindex 1: Model coefficients
MODEL_COEFFICIENTS = {
    "crosses": 0.519,
    "dribbles": 0.118,
    "passes": 0.034,
    "opp_interceptions": -0.024,
    "opp_yellows": 0.253,
    "opp_reds": 1.023,
    "opp_tackle_win_ratio": -0.170,
    "opp_clearances": -0.017,
    "constant": 6.463,
}

## Subindex 2
POINTS_FOR_WIN = 3
POINTS_FOR_DRAW = 1
POINTS_FOR_LOSS = 0

## Subindex 3
POINTS_PER_GAME = 1.34

## Subindex 4
POINTS_PER_GOAL = 1.039

## Subindex 5
POINTS_PER_ASSIST = 1.039

## Subindex 6
POINTS_PER_CLEAN_SHEET_GOALKEEPER = 0.585
POINTS_PER_CLEAN_SHEET_DEFENDER = 0.364
POINTS_PER_CLEAN_SHEET_MIDFIELDER = 0.150
POINTS_PER_CLEAN_SHEET_STRIKER = 0.071

In [20]:
# Necessary variables for the index with dummy values
position = "ST"
minutes_played = 90
goals = 2
assists = 1

home_goals = 4
away_goals = 2

team_minutes = 990
clean_sheets = 0

stats = {
    "crosses": 3,
    "dribbles": 1,
    "passes": 20,
    "opp_interceptions": 3,
    "opp_yellows": 1,
    "opp_reds": 0,
    "opp_tackle_win_ratio": 0.2,
    "opp_clearances": 1,
}

## The Index

### Subindex 1: Modelling Match Outcome

In [21]:
index_1 = MODEL_COEFFICIENTS["constant"]
for action, coeff in MODEL_COEFFICIENTS.items():
    if action != "constant":
        count = stats.get(action, 0)
        index_1 += coeff * count

### Subindex 2: Points-Sharing Index

In [22]:
if home_goals > away_goals:
    index_2 = (minutes_played / team_minutes) * POINTS_FOR_WIN
elif home_goals == away_goals:
    index_2 = (minutes_played / team_minutes) * POINTS_FOR_DRAW
else:  # home_goals < away_goals
    index_2 = (minutes_played / team_minutes) * POINTS_FOR_LOSS

### Subindex 3: Appearance Index

In [23]:
index_3 = (minutes_played / team_minutes) * POINTS_PER_GAME

### Subindex 4: Goal-Scoring Index

In [24]:
index_4 = goals * POINTS_PER_GOAL

### Subindex 5: Assists Index

In [25]:
index_5 = assists * POINTS_PER_ASSIST

### Subindex 6: Clean-Sheets Index

In [26]:
if clean_sheets:
    if position == "goalkeeper":
        index_6 = POINTS_PER_CLEAN_SHEET_GOALKEEPER
    elif position == "defender":
        index_6 = POINTS_PER_CLEAN_SHEET_DEFENDER
    elif position == "midfielder":
        index_6 = POINTS_PER_CLEAN_SHEET_MIDFIELDER
    elif position == "striker":
        index_6 = POINTS_PER_CLEAN_SHEET_STRIKER
else:
    index_6 = 0

## Final Index

In [27]:
final_index = 100 * (
    0.25 * index_1 + 0.375 * index_2 + 0.125 * index_3 + 0.125 * index_4 + 0.0625 * index_5 + 0.0625 * index_6
)

In [28]:
round(final_index, 2)

267.92